In [ ]:
import os
import pandas as pd
from openai import OpenAI
from datasets import load_dataset
import openpyxl
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import requests, tempfile, pathlib

import asyncio
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
from ragas.dataset_schema import SingleTurnSample

from sentence_transformers import CrossEncoder

import nest_asyncio
nest_asyncio.apply()

In [ ]:
ds = load_dataset("PatronusAI/financebench")

In [8]:
ds.column_names

{'train': ['financebench_id',
  'company',
  'doc_name',
  'question_type',
  'question_reasoning',
  'domain_question_num',
  'question',
  'answer',
  'justification',
  'dataset_subset_label',
  'evidence',
  'gics_sector',
  'doc_type',
  'doc_period',
  'doc_link']}

In [3]:
train_ds = ds["train"]

# filter out metrics-generated
train_ds = train_ds.filter(
    lambda x: x["question_type"] != "metrics-generated"
)

In [16]:
train_ds.to_pandas()["question_type"].value_counts()

question_type
domain-relevant    50
novel-generated    50
Name: count, dtype: int64

In [4]:
REPO_BASE = "https://github.com/patronus-ai/financebench/tree/main/pdfs/"

doc_names = set(train_ds["doc_name"])
doc_index = {doc_name: REPO_BASE + doc_name for doc_name in doc_names}

def resolve_link(example):
    example["doc_link"] = doc_index.get(
        example["doc_name"],
        None  # fallback if missing
    )
    return example

train_ds = train_ds.map(resolve_link)

Q1

In [ ]:
df = train_ds.to_pandas()
df = df.sort_values("financebench_id")

sample = (
    df.groupby("question_type", group_keys=False)
    .apply(lambda g: g.head(5))
    .reset_index(drop=True)
)

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)

MODEL = "meta-llama/Llama-3.3-70B-Instruct"

SYSTEM_PROMPT = """You are a financial analyst assistant.
Answer the user's question as accurately as possible using your knowledge.
If you do not have enough information to answer confidently, say so clearly. 
Do not fabricate numbers or facts."""

def ask_model(question: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0.0,       # deterministic
    )
    return response.choices[0].message.content.strip()

In [ ]:
results = []

for _, row in sample.iterrows():

    model_answer = ask_model(row["question"])

    results.append({
        "financebench_id":  row["financebench_id"],
        "question_type":    row["question_type"],
        "question":         row["question"],
        "naive_answer":     model_answer,
        "ground_truth":      row["answer"],
        "verdict":          "",
    })

results_df = pd.DataFrame(results)

results_df.to_excel("assignment2_naive_generation.xlsx", index=False)

The model was quite successful at the domain relevant questions, returning correct conclusions with some divergance in the exact details. On the other hand, for novel generated questions the model was either wrong or refused to answer, claiming it needs more information. This difference indicates that for domain relevant questions, the model already knows the answer, meaning the model has seen it during training and therfore it can answer confidently. On the contrary, novel generated questions are designed to be novel, meaning the answers need to be derived and are not stored in memory. The model is missing both the details needed to answer and the capacity to derive an answer, due to the retrieval burden from such a large context.

Q2

Indexing converts the raw documents into a searchable numerical format. It contributes to the pipeline by creating a map of the data, where similar concepts are stored close together in a vector space. Failures often occur during chunking (ex: splitting a sentence in half and as a result losing the context) or when using an embedding model that doesn't understand the domain's specific jargon. This is an offline process that typically happens once.

Retrieval is the searching of the vector store to find the most relevant chunks of information. It ensures the model has the specific facts needed to answer the prompt rather than relying on its internal training data alone. It can fail due to semantic mismatch, ex: retrieving a document about Apple (the tech company) when the user asked about apple (the fruit). This process happens per query in real time.

Generation is the creation of a coherent, natural language answer from the retrieved snippets and the original user query. Its role is to act as a reasoning engine that filters out noise from the retrieved data to provide a direct response. It can fail when the model ignores the retrieved facts and confidently makes up an incorrect answer (hallucination). This happens per query.

Q3

In [6]:
# Download PDFs
RAW_BASE  = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/"
PDF_DIR = pathlib.Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)

def download_pdf(doc_name: str) -> pathlib.Path:
    doc_name = doc_name + ".pdf" if not doc_name.endswith(".pdf") else doc_name
    dest = PDF_DIR / doc_name
    if dest.exists():
        return dest                          # skip if already downloaded
    url = RAW_BASE + doc_name
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    return dest

print(f"Downloading {len(doc_names)} PDFs...")
pdf_paths = {}
for name in doc_names:
    try:
        pdf_paths[name] = download_pdf(name)
    except Exception as e:
        print(f"  X {name}: {e}")

print(f"Downloaded {len(pdf_paths)} PDFs successfully.")

Downloaded 42 PDFs successfully.


In [ ]:
# Load PDFs with PyPDFLoader + attach metadata 
# Build a lookup doc_name: {company, doc_period} from the dataframe
doc_meta_lookup = (
    df[["doc_name", "company", "doc_period"]]
    .drop_duplicates("doc_name")
    .set_index("doc_name")
    .to_dict("index")
)

all_pages = []

for doc_name, pdf_path in pdf_paths.items():
    try:
        loader = PyPDFLoader(str(pdf_path))
        pages  = loader.load()               # one Document per page
    except Exception as e:
        print(f"  X Could not load {doc_name}: {e}")
        continue

    meta = doc_meta_lookup.get(doc_name, {})

    for page in pages:
        # PyPDFLoader sets page (0-indexed) — copy to page_number for clarity
        page.metadata["doc_name"]    = doc_name
        page.metadata["company"]     = meta.get("company", "")
        page.metadata["doc_period"]  = meta.get("doc_period", "")
        page.metadata["page_number"] = page.metadata.get("page", 0)

    all_pages.extend(pages)

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(all_pages)

# Embed & store in FAISS 
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},          # change to "cuda" if available
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("vectorstore")

In [ ]:
# Pick 3 questions with known evidence for a meaningful check
sample_questions = (
    df[df["evidence"].apply(lambda e: len(e) > 0)]   # has evidence
    .drop_duplicates("doc_name")                      # spread across docs
    .head(3)
)

TOP_K = 5

retrieval_records = []

for _, row in sample_questions.iterrows():
    query        = row["question"]
    expected_doc = row["doc_name"]
    evidence     = row["evidence"]       # list of dicts with evidence_page_num, evidence_text
    # run through the evidence to extract what we expect to retrieve
    expected_pages = [e["evidence_page_num"] for e in evidence]
    evidence_texts = [e["evidence_text"]     for e in evidence]

    # return the top-K most similar chunks for the question
    # returns a list of Documents with meetadata and page_content
    retrieved = vectorstore.similarity_search(query, k=TOP_K)

    for rank, chunk in enumerate(retrieved, start=1):
        m = chunk.metadata
        retrieval_records.append({
            # question context 
            "financebench_id":   row["financebench_id"],
            "question_type":     row["question_type"],
            "question":          query,
            # expected 
            "expected_doc_name": expected_doc,
            "expected_pages":    expected_pages,
            "evidence_texts":    evidence_texts,
            # actual retrieval
            "rank":              rank,
            "retrieved_doc":     m.get("doc_name", ""),
            "retrieved_page":    m.get("page_number", -1),
            "chunk_text":        chunk.page_content,
            # boolean checks of whether we got the expected doc and page in this chunk
            "doc_match":         m.get("doc_name", "") == expected_doc,
            "page_match":        m.get("page_number", -1) in expected_pages,
        })

retrieval_df = pd.DataFrame(retrieval_records)
retrieval_df.to_csv("task3_retrieval_check.csv", index=False)

I sampled top 5 retrievals for 3 questions. For all of them, the top 2 ranked retrievals retrieved the correct document. In addition, more of the top 5 also retrieved the correct document. (3 out 5, 4 out of 5, 5 out of 5.) Only 1 of all 15 retrievals retrieved the correct page. Overall the retrieved chuncks do not contain the evidence and it is unclear what the model thought the connection was between the retrieved chunks and the question.

Q4

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True,
)

MODEL = "meta-llama/Llama-3.3-70B-Instruct"

SYSTEM_PROMPT = """\
You are a precise financial analyst assistant.

Rules you must follow:
1. Answer ONLY using the context passages provided below the user's question.
2. If the answer is not present in the context, say exactly:
   "The provided context does not contain enough information to answer this question."
   Do not guess, infer beyond what is written, or use outside knowledge.
3. Keep your answer concise and to the point.
4. Every factual claim must be followed by a citation in the form [doc_name, p.N],
   where N is the page number of the chunk it came from.
"""

def format_context(chunks: list) -> str:
    """
    Formats retrieved chunks using a clear separator.
    Handles empty retrievals with a notice string.
    """
    if not chunks:
        return "<<< NO CONTEXT RETRIEVED >>>"

    parts = []
    for i, chunk in enumerate(chunks, start=1):
        doc  = chunk.metadata.get("doc_name", "unknown")
        page = chunk.metadata.get("page_number", "?")
        parts.append(
            f"--- Chunk {i} | doc_name: {doc} | page: {page} ---\n"
            f"{chunk.page_content.strip()}\n"
            f"--- End Chunk {i} ---"
        )
    return "\n\n".join(parts)


def answer_with_rag(query: str, k: int = 4) -> dict:
    """
    Retrieve the top-k chunks from the vector store, 
    then generate an answer grounded strictly in those chunks.

    Returns
    -------
    {
        "answer":            str,
        "retrieved_chunks":  list of dicts with keys:
                               chunk_text, doc_name, page_number
    }
    """
    # return the top-K most similar chunks for the question
    # returns a list of Documents with meetadata and page_content
    chunks = vectorstore.similarity_search(query, k=k)

    # format context
    context_block = format_context(chunks)

    user_message = (
        f"Question: {query}\n\n"
        f"Context:\n{context_block}"
    )

    # generate RAG answer
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.0,
    )

    answer = response.choices[0].message.content.strip()

    # retrieved chunks for inspection
    retrieved_chunks = [
        {
            "chunk_text":   chunk.page_content.strip(),
            "doc_name":     chunk.metadata.get("doc_name", "unknown"),
            "page_number":  chunk.metadata.get("page_number", -1),
        }
        for chunk in chunks
    ]

    return {
        "answer":           answer,
        "retrieved_chunks": retrieved_chunks,
    }

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2482.86it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q5

In [21]:
# 10-question sample from Task 1
results_df = pd.read_excel("assignment2_naive_generation.xlsx")

rag_rows = []

for _, row in results_df.iterrows():
    print(f"Running RAG for [{row['financebench_id']}]...")
    rag_result = answer_with_rag(row["question"], k=4)

    rag_rows.append({
        "financebench_id": row["financebench_id"],
        "question_type":   row["question_type"],
        "question":        row["question"],
        "naive_answer":    row["naive_answer"],
        "RAG_answer":      rag_result["answer"],
        "ground_truth":    row["ground_truth"],
    })

compare_df = pd.DataFrame(rag_rows)

COLS = ["financebench_id", "question_type", "question",
        "naive_answer", "RAG_answer", "ground_truth"]

# reorder columns to match original layout
compare_df = compare_df[COLS]

output_file = "assignment2_run_and_compare.xlsx"
compare_df.to_excel(output_file, index=False)

Running RAG for [financebench_id_00005]...
Running RAG for [financebench_id_00070]...
Running RAG for [financebench_id_00080]...
Running RAG for [financebench_id_00206]...
Running RAG for [financebench_id_00215]...
Running RAG for [financebench_id_00283]...
Running RAG for [financebench_id_00288]...
Running RAG for [financebench_id_00299]...
Running RAG for [financebench_id_00302]...
Running RAG for [financebench_id_00382]...


For 7 out of 10 questions, RAG refused to answer. The 3 questions it answered were correct. 
Comparing to the naive model from task 1:

For domain relevant questions, the naive model was quite successful (correct or partially correct) and the RAG model mostly refused to answer. This is a clear indication that the retrievals were problematic. In this case, RAG made things worse. The model had knowledge in memory, but RAG grounded the model with irrelevant context.

For novel generated questions, the naive model generally was wrong or refused (info not in memory). The RAG model also refused for 3 questions (info not in retrieved chunks) but for both answers it was willing to give - they were correct! Over here, retrieval found the right information and the model had the capacity to derive the answer correctly.

Q6

In [18]:
# Run ALL questions through RAG pipeline

df_sorted = df.sort_values("financebench_id").reset_index(drop=True)

all_results = []
for _, row in df_sorted.iterrows():
    print(f"RAG: [{row['financebench_id']}]")
    rag_result = answer_with_rag(row["question"], k=5)   # k=5 covers page_hit@5
    all_results.append({
        "financebench_id":   row["financebench_id"],
        "question_type":     row["question_type"],
        "question":          row["question"],
        "ground_truth":      row["answer"],
        "rag_answer":        rag_result["answer"],
        "retrieved_chunks":  rag_result["retrieved_chunks"],
        "evidence":          row["evidence"],
    })

rag_df = pd.DataFrame(all_results)

RAG: [financebench_id_00005]
RAG: [financebench_id_00070]
RAG: [financebench_id_00080]
RAG: [financebench_id_00206]
RAG: [financebench_id_00215]
RAG: [financebench_id_00216]
RAG: [financebench_id_00222]
RAG: [financebench_id_00283]
RAG: [financebench_id_00288]
RAG: [financebench_id_00299]
RAG: [financebench_id_00302]
RAG: [financebench_id_00382]
RAG: [financebench_id_00394]
RAG: [financebench_id_00407]
RAG: [financebench_id_00438]
RAG: [financebench_id_00460]
RAG: [financebench_id_00464]
RAG: [financebench_id_00476]
RAG: [financebench_id_00494]
RAG: [financebench_id_00499]
RAG: [financebench_id_00517]
RAG: [financebench_id_00521]
RAG: [financebench_id_00540]
RAG: [financebench_id_00552]
RAG: [financebench_id_00563]
RAG: [financebench_id_00566]
RAG: [financebench_id_00585]
RAG: [financebench_id_00591]
RAG: [financebench_id_00601]
RAG: [financebench_id_00603]
RAG: [financebench_id_00605]
RAG: [financebench_id_00606]
RAG: [financebench_id_00651]
RAG: [financebench_id_00669]
RAG: [financeb

In [26]:
rag_df.to_csv("task6_rag_df.csv", index=False)

In [19]:
# correctness
JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.2"

judge_client = AsyncOpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

JUDGE_SYSTEM = """\
You are a strict financial-answer evaluator.
Given a question, a ground-truth answer, and a model answer, 
decide whether the model answer is correct.

Rules:
- Correct  = the model answer conveys the same key fact(s) as the ground truth
  (minor wording differences and rounding within 1% are acceptable).
- Incorrect = the model answer is wrong, refuses to answer, 
  or is missing critical information present in the ground truth.

Respond with EXACTLY two lines:
verdict: correct   OR   verdict: incorrect
reason: <one sentence>"""

# judge one question-answer pair
async def judge_one(question: str, ground_truth: str, model_answer: str) -> dict:
    # non blocking call frees the event loop to run other judge_one calls in the meantime
    # the coroutine waiting for the API response is suspended and does not block others from running concurrently 
    resp = await judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content":
                f"Question: {question}\n"
                f"Ground truth: {ground_truth}\n"
                f"Model answer: {model_answer}"},
        ],
        temperature=0.0,
        max_tokens=120,
    )
    raw = resp.choices[0].message.content.strip()
    verdict = "incorrect"
    reason  = raw
    for line in raw.splitlines():
        if line.lower().startswith("verdict:"):
            verdict = "correct" if line.lower().startswith("verdict: correct") else "incorrect"
        if line.lower().startswith("reason:"):
            reason = line.split(":", 1)[1].strip()
    return {"verdict": verdict, "reason": reason}

async def run_correctness(df: pd.DataFrame) -> list:
    tasks = [
        judge_one(r["question"], r["ground_truth"], r["rag_answer"])
        for _, r in df.iterrows()
    ]
    # suspend until all judge_one coroutines complete concurrently, 
    # then return all results together
    return await asyncio.gather(*tasks)

# event loop sending all judgments concurrently and waiting for results
correctness_results = asyncio.run(run_correctness(rag_df))

rag_df["correctness"] = [1 if r["verdict"] == "correct" else 0 for r in correctness_results]
rag_df["correctness_reason"] = [r["reason"] for r in correctness_results]

rag_df.to_csv("task6_rag_df.csv", index=False)

In [20]:
# faithfullness
ragas_client = AsyncOpenAI( 
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

ragas_llm = llm_factory(
    model=JUDGE_MODEL,
    client=ragas_client,
)

faithfulness_metric = Faithfulness(llm=ragas_llm)

FAITH_IDS = set(
    rag_df["financebench_id"].head(20).tolist() # already sorted by financebench_id above
)

async def compute_faithfulness(row: dict) -> float:
    """Score one row synchronously using Ragas .score()"""
    context_strings = [c["chunk_text"] for c in row["retrieved_chunks"]]
    try:
        score = await faithfulness_metric.ascore (
            user_input=row["question"], 
            response=row["rag_answer"], 
            retrieved_contexts=context_strings,
        )
        return score.value if score is not None else float("nan")
    except Exception as e:
        print(f"  Faithfulness error [{row['financebench_id']}]: {e}")
        return float("nan")


rag_df["faithfulness"] = float("nan")

async def run_faithfulness(df: pd.DataFrame) -> list:
    mask = df["financebench_id"].isin(FAITH_IDS)
    results = []
    for _, row in df[mask].iterrows():
        results.append( await compute_faithfulness(row.to_dict()))
    return results

faithfulness_results = await run_faithfulness(rag_df)
faith_mask = rag_df["financebench_id"].isin(FAITH_IDS)
rag_df.loc[faith_mask, "faithfulness"] = faithfulness_results
rag_df.to_csv("task6_rag_df.csv", index=False)


In [22]:
import numpy as np

In [23]:
# retrieval hit-rate (page_hit@k)
def get_evidence_pages(evidence) -> set:
    """Return the set of all evidence page numbers for a question."""
    if evidence is None:
        return set()
    # evidence is a list of dicts: [{evidence_page_num: int, ...}, ...]
    if isinstance(evidence, np.ndarray):
        evidence = evidence.tolist()
    if len(evidence) == 0:
        return set()
    pages = set()
    for item in evidence:
        if isinstance(item, dict) and "evidence_page_num" in item:
            pages.add(item["evidence_page_num"])
    return pages

def page_hit_at_k(retrieved_chunks: list, evidence_pages: set, k: int) -> int:
    """1 if any of the top-k chunks comes from an evidence page, else 0."""
    if not evidence_pages:
        return float("nan")          # can't evaluate without ground truth
    top_k = retrieved_chunks[:k]
    retrieved_pages = {c["page_number"] for c in top_k}
    return 1 if retrieved_pages & evidence_pages else 0

K_VALUES = [1, 3, 5]

for k in K_VALUES:
    col = f"page_hit_at_{k}"
    rag_df[col] = rag_df.apply(
        lambda r: page_hit_at_k(
            r["retrieved_chunks"],
            get_evidence_pages(r["evidence"]),
            k,
        ),
        axis=1,
    )

rag_df.to_csv("task6_rag_df.csv", index=False)

In [24]:
# per-question xlsx file
csv_cols = (
    ["financebench_id", "question", "correctness", "faithfulness"]
    + [f"page_hit_at_{k}" for k in K_VALUES]
)

xlsx_df = rag_df[csv_cols].copy()
xlsx_df.to_excel("assignment2_evaluation.xlsx", index=False)

# aggregate numbers
agg = {
    "avg_correctness":   rag_df["correctness"].mean(),
    "avg_faithfulness":  rag_df["faithfulness"].mean(skipna=True),   # 20-row subset
}
for k in K_VALUES:
    col = f"page_hit_at_{k}"
    agg[f"page_hit_at_{k}"] = rag_df[col].mean(skipna=True)

agg_df = pd.DataFrame([agg])
print(agg_df.to_string(index=False))

 avg_correctness  avg_faithfulness  page_hit_at_1  page_hit_at_3  page_hit_at_5
            0.37          0.448333           0.21           0.34           0.43


Q7

In [29]:
async def evaluate_results(results_rows):
    """
    Given a list of result dicts, compute all three metrics.
    results_rows: list of dicts with keys:
        financebench_id, question, ground_truth, rag_answer, retrieved_chunks, evidence
    Returns dict of aggregate scores.
    """
    rdf = pd.DataFrame(results_rows)

    # correctness
    # correctness_results = asyncio.run(run_correctness(rdf))
    correctness_results = await run_correctness(rdf)
    rdf["correctness"] = [1 if r["verdict"] == "correct" else 0 for r in correctness_results]

    # faithfulness — fixed 20-question subsample only
    rdf["faithfulness"] = float("nan")
    faithfulness_results = await run_faithfulness(rdf)
    faith_mask = rdf["financebench_id"].isin(FAITH_IDS)
    rdf.loc[faith_mask, "faithfulness"] = faithfulness_results

    # page hit-rate
    for k in K_VALUES:
        rdf[f"page_hit_at_{k}"] = rdf.apply(
            lambda r: page_hit_at_k(
                r["retrieved_chunks"],
                get_evidence_pages(r["evidence"]),
                k,
            ), axis=1
        )

    agg = {
        "correctness":   rdf["correctness"].mean(),
        "faithfulness":  rdf["faithfulness"].mean(skipna=True),
    }
    for k in K_VALUES:
        agg[f"page_hit_at_{k}"] = rdf[f"page_hit_at_{k}"].mean(skipna=True)

    return agg, rdf

In [30]:
# baseline from task 6

# turn each row into a standalone dictionary
# baseline_rows = rag_df.to_dict("records")
baseline_agg  = {
    "correctness":   rag_df["correctness"].mean(),
    "faithfulness":  rag_df["faithfulness"].mean(skipna=True),
    **{f"page_hit_at_{k}": rag_df[f"page_hit_at_{k}"].mean(skipna=True)
       for k in K_VALUES},
}

Exp 1: stricter prompt

Hypothesis: the baseline prompt's instruction to stay grounded is correct but vague. Adding an explicit instruction to extract exact figures and show calculation steps should improve correctness on numerical questions while keeping faithfulness stable, since the model is still bound to context.

In [31]:
STRICT_SYSTEM_PROMPT = """\
You are a precise financial analyst assistant.

Rules you must follow:
1. Answer ONLY using the context passages provided.
2. If the answer is not present in the context, say exactly:
   "The provided context does not contain enough information to answer this question."
   Do not guess, infer beyond what is written, or use outside knowledge.
3. For numerical questions: quote the exact figure(s) from the context, 
    then show any calculation steps explicitly before giving the final answer.
4. Keep answers concise. Every factual claim must be cited as [doc_name, p.N],
    where N is the page number of the chunk it came from.
"""

def answer_with_rag_exp1(query: str, k: int = 4) -> dict:
    chunks = vectorstore.similarity_search(query, k=k)

    context_block = format_context(chunks) 

    user_message = (
        f"Question: {query}\n\n"
        f"Context:\n{context_block}"
    )
    response = client.chat.completions.create(   # sync client from Task 4
        model=MODEL,
        messages=[
            {"role": "system", "content": STRICT_SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.0,
    )

    answer = response.choices[0].message.content.strip()

    retrieved_chunks = [
        {
            "chunk_text":   chunk.page_content.strip(),
            "doc_name":     chunk.metadata.get("doc_name", "unknown"),
            "page_number":  chunk.metadata.get("page_number", -1),
        }
        for chunk in chunks
    ]

    return {
        "answer":           answer,
        "retrieved_chunks": retrieved_chunks,
    }


exp1_rows = []
for _, row in df_sorted.iterrows():
    exp1_res = answer_with_rag_exp1(row["question"], k=5)
    exp1_rows.append({
        "financebench_id": row["financebench_id"],
        "question":        row["question"],
        "ground_truth":    row["answer"],
        "rag_answer":      exp1_res["answer"],
        "retrieved_chunks":exp1_res["retrieved_chunks"],
        "evidence":        row["evidence"],
    })

exp1_agg, exp1_df = await evaluate_results(exp1_rows)

Exp 2: larger k  (k=8)

Hypothesis: raising k from 5 to 8 gives the model a better chance of seeing the evidence page, which should improve page-hit@k and correctness. Faithfulness may drop slightly because more irrelevant chunks enter the context.

In [ ]:
exp2_rows = []
for _, row in df_sorted.iterrows():
    exp2_res = answer_with_rag_exp1(row["question"], k=8)
    exp2_rows.append({
        "financebench_id": row["financebench_id"],
        "question":        row["question"],
        "ground_truth":    row["answer"],
        "rag_answer":      exp2_res["answer"],
        "retrieved_chunks":exp2_res["retrieved_chunks"],
        "evidence":        row["evidence"],
    })

exp2_agg, exp2_df = await evaluate_results(exp2_rows)

Exp 3: reranker  (retrieve top-20, rerank to top-4)

Hypothesis: FAISS cosine similarity often misranks chunks because it compares query and chunk embeddings independently. A cross-encoder sees the query and chunk together, so reranking should surface the evidence page more reliably, improving page-hit@k and correctness without changing faithfulness much.

In [35]:

reranker = CrossEncoder("BAAI/bge-reranker-base")

def answer_with_rag_reranked(query: str, fetch_k: int = 20, final_k: int = 4) -> dict:
    # first pass: broad FAISS retrieval
    candidates = vectorstore.similarity_search(query, k=fetch_k)

    # second pass: cross-encoder reranking
    pairs  = [(query, c.page_content) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    chunks = [doc for _, doc in ranked[:final_k]]

    context_block = format_context(chunks)

    user_message = (
        f"Question: {query}\n\n"
        f"Context:\n{context_block}"
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.0,
    )
    answer = response.choices[0].message.content.strip()
    retrieved_chunks = [
        {
            "chunk_text":   chunk.page_content.strip(),
            "doc_name":     chunk.metadata.get("doc_name", "unknown"),
            "page_number":  chunk.metadata.get("page_number", -1),
        }
        for chunk in chunks
    ]

    return {
        "answer":           answer,
        "retrieved_chunks": retrieved_chunks,
    }



exp3_rows = []
for _, row in df.sort_values("financebench_id").iterrows():
    exp3_res = answer_with_rag_reranked(row["question"])
    exp3_rows.append({
        "financebench_id": row["financebench_id"],
        "question":        row["question"],
        "ground_truth":    row["answer"],
        "rag_answer":      exp3_res["answer"],
        "retrieved_chunks":exp3_res["retrieved_chunks"],
        "evidence":        row["evidence"],
    })

exp3_agg, exp3_df = await evaluate_results(exp3_rows)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1783.92it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
experiments = [
    {
        "experiment": "Baseline",
        "change":     "Task 6 defaults",
        **baseline_agg,
    },
    {
        "experiment": "Exp 1: Stricter prompt",
        "change":     "Added explicit step-show instruction for numerical answers",
        **exp1_agg,
    },
    {
        "experiment": "Exp 2: Larger k",
        "change":     "k raised from 4 to 8",
        **exp2_agg,
    },
    {
        "experiment": "Exp 3: Reranker",
        "change":     "FAISS top-20, rerank and select top-4",
        **exp3_agg,
    },
]

results_df = pd.DataFrame(experiments)
results_df.to_excel("assignment2_improvement_cycles.xlsx", index=False)

In [37]:
results_df.to_excel("assignment2_improvement_cycles.xlsx", index=False)

Interpretation

Exp 1: Stricter prompt

The stricter prompt hurt correctness while improving faithfulness indicating that the model became more conservative, grounding itself more tightly in the context but refusing more often when the exact figures weren't explicitly stated. The prompt made the model more faithful but less willing to commit to an answer, which penalises correctness on questions that require some inference.

Exp 2: Larger k (k=8)

Both correctness and faithfulness improved, which is the most encouraging result across all experiments. More context didn't distract the model as hypothesised, rather it gave the model enough material to both ground its answer and derive correct results, suggesting the baseline k=5 was simply leaving relevant chunks out and the stricter prompt (exp 1) was limiting.

Exp 3: Reranker

Correctness improved, but page-hit@k dropped which is counterintuitive. The cross-encoder appears to be reranking toward chunks that are semantically fluent and useful for generation, but not necessarily from the exact evidence page the dataset designates as ground truth. The reranker improves answer quality while actually moving away from the gold evidence pages, suggesting the dataset's evidence pages and the model's preferred context don't always align. Faithfulness also dropped and this may be a result of inference or calculation between the retrieved evidence and the final answer, which is actually a good thing.

Wrap up

The pipeline fails most at retrieval (page hit), but the failure is nuanced. Experiment 3 shows that a sophisticated reranker can improve answer quality while missing the designated evidence pages, meaning retrieval is finding useful content even if it's not the right content as defined by the dataset.

Experiment 2's result is the clearest signal: simply giving the model more chunks improved both correctness and faithfulness, which means the baseline retrieval was consistently leaving relevant material just outside the top-5 cutoff. When the right context is present the model can reason over it correctly and produce better answers. This matches the conclusion the novel-generated successes showed in task 5.

If given one more week, I would experiment with metadata-filtered retrieval: since every question in the dataset is tied to a specific company and filing period, restricting FAISS search to chunks from the matching doc_name before ranking would eliminate cross-document noise entirely, likely improving both page-hit@k and correctness without any change to the generator.